# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`
This notebook guides you through loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Name:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, 'version', ''))
print("Published:", getattr(metadata, 'datePublished', ''))


## 2. Data Overview
Review available record sets and field IDs. Each entity is referenced by its `@id`.

We first list all record sets and for each, fields and columns (using their `@id`).

In [ ]:
# List all record sets, fields, and columns by `@id`
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '')}")
    if 'field' in rs:
        print(f"  Fields:")
        for field in rs['field']:
            fid = field['@id'] if isinstance(field, dict) else field
            print(f"    - {fid}")
    if 'column' in rs:
        print(f"  Columns:")
        for col in rs['column']:
            cid = col['@id'] if isinstance(col, dict) else col
            print(f"    - {cid}")
    print('')

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field/column `@id`s shown above.


In [ ]:
# Prepare record set IDs for direct extraction using their `@id`
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        # Load records using mlcroissant (returns dicts for each record)
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records for RecordSet {record_set_id}")
    except Exception as e:
        print(f"Could not load RecordSet {record_set_id}: {e}")

# For demonstration, pick the first record set for further analysis
if record_set_ids:
    main_rs = record_set_ids[0]
    print(f"\nColumns in record set {main_rs}:\n", dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head(5))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data for summary statistics.

Let's select a numeric field (e.g., a GAD-7 or PHQ-9 score field), filter on it, normalize, and group by another field (e.g., `gender` or similar). All references use their `@id` as shown in previous outputs.


In [ ]:
# Automatic field selection for demo; override manually for best results
# Suggestion: replace with actual field @id if known, e.g. 'gad7_score', 'phq9_score'
import numpy as np

df = dataframes[main_rs]
# Try to pick a numeric column
numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_candidates:
    # Try to coerce likely score columns to numeric
    for col in df.columns:
        if 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower():
            df[col] = pd.to_numeric(df[col], errors='coerce')
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field for analysis: {numeric_field}")
else:
    raise ValueError('No numeric field found for EDA.')

threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} rows")
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nSample of normalized {numeric_field}:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (e.g., 'gender' or 'level_of_education')
group_candidates = [col for col in df.columns if col != numeric_field and df[col].nunique() < 10 and df[col].dtype == 'O']
group_field = group_candidates[0] if group_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
    print(f"\nMean {numeric_field} by {group_field}:")
    display(grouped_df)
else:
    print('No suitable group field found for grouping analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields. We'll plot the distribution of the selected numeric field and optionally by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

if group_field:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
- This notebook demonstrated how to load, explore, and process a dataset defined by a Croissant schema, referencing all datasets, fields, and columns by their `@id`.
- We performed basic EDA, normalized scores, and observed group differences, e.g., by gender or education level, using Python tools.
- You can extend this analysis by exploring additional record sets, applying more sophisticated analysis, or building machine learning models on the extracted DataFrames.